This notebook extends the ["Parallel Execution (Part 1)"](./Parallel%20Execution%20%28Part%201%29.ipynb) example.

In [ ]:
# Install LangGraph
!pip install -q langgraph

In [ ]:
import operator  # Provides operator.add (list accumulation) and operator.or_ (set union) for reducers
import time

from IPython.display import Image
from google.colab import userdata
from langchain_core.runnables import Runnable, RunnableConfig
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.state import CompiledStateGraph
from pathlib import Path
from typing import Annotated, TypedDict

# Helper: renders the compiled graph as a PNG and shows it inline
def display_graph(runnable: Runnable, output_png: Path) -> None:
    with output_png.open(mode="wb") as file:
        file.write(runnable.get_graph().draw_mermaid_png())

    display(Image(output_png, format="png"))

# Helper: prints the graph state at each execution step
def explore_state_history(compiled_state_graph: CompiledStateGraph, config: RunnableConfig):
    state_history = list(compiled_state_graph.get_state_history(config))

    for snapshot in reversed(state_history):
        print(f"Step: {snapshot.metadata['step']}")
        print("Current state:")
        print(snapshot.values)
        print(f"Next: {snapshot.next}")
        print()

In [ ]:
checkpointer = InMemorySaver()

# Custom reducer for 'findings' — merges two dicts produced by parallel nodes, 'b' wins on conflicts
def merge_findings(a: dict[str, str], b: dict[str, str]) -> dict[str, str]:
    # `b` overwrites `a` in case of conflicts.
    # This fragment can be modified to raise an error instead.
    return {**a, **b}

# Extended state — adds verification tracking on top of findings/summary from Part 1
class ResearchState(TypedDict):
    topic: str
    findings: Annotated[dict[str, str], merge_findings]         # All source results, merged
    verification_requests: Annotated[list[str], operator.add]   # Sources needing verification (lists accumulate)
    verified: Annotated[set[str], operator.or_]                 # Approved sources (sets union-merge)
    blocked: Annotated[set[str], operator.or_]                  # Rejected sources (sets union-merge)
    summary: str                                                 # Final summary

In [ ]:
# --- Parallel data-fetching nodes (same as Part 1 but now they also request verification) ---

def wikipedia(state: ResearchState):
    print("[WIKIPEDIA] node is executing")
    time.sleep(2)
    # wikipedia requests verification — adds "wikipedia" to the verification_requests list
    return { "findings": { "wiki": f"[WIKIPEDIA] {state['topic']}" }, "verification_requests": ["wikipedia"] }

def news(state: ResearchState):
    print("[LIVE NEWS] node is executing")
    time.sleep(2)
    # news also requests verification
    return { "findings": { "news": f"[LIVE NEWS] Sensational! {state['topic']}" }, "verification_requests": ["news"] }

def arxiv(state: ResearchState):
    print("[ARXIV] node is executing")
    time.sleep(2)
    # arxiv does NOT request verification — goes directly to merge
    return { "findings": { "arxiv": f"[ARXIV] The theoretical hyperchaotic entanglement of relativities related to \"{state['topic']}\"" } }

# --- Automated verifier node (no human input — uses a deterministic rule instead) ---
def verify(state: ResearchState):
    # NEW in Part 2: verification is AUTOMATIC (not human-in-the-loop like in Interrupts.ipynb).
    # The rule: if the source name has an odd number of characters -> verified, even -> blocked.
    current_verified = state.get('verified', set())
    current_blocked = state.get('blocked', set())

    new_verified = set()
    new_blocked = set()

    for request in state.get('verification_requests', {}):
        # Skip already-decided sources
        if request in current_verified or request in current_blocked:
            continue

        # Deterministic rule: odd-length name = verified, even-length = blocked
        if len(request) % 2 == 0:
            new_blocked.add(request)
        else:
            new_verified.add(request)

    return { "verified": new_verified, "blocked": new_blocked }

def _annotate_finding(finding_key: str, state: ResearchState):
    # Helper: adds a "(verified)" or "(blocked)" annotation to the summary
    if finding_key in state.get('verified', set()):
        return " (verified)"
    elif finding_key in state.get('blocked', set()):
        return " (blocked)"
    else:
        return ""

def merge(state: ResearchState):
    # Guard: only produce a summary once all requested verifications have been resolved
    if len(state.get('verification_requests', [])) > len(state.get('verified', set())) + len(state.get('blocked', set())):
        print("There are pending verification requests. Merge node will be skipped.")
        return {}

    # Build the final summary with annotation labels
    bullets = "\n".join(f" - {finding}{_annotate_finding(key, state)}" for key, finding in state.get("findings", {}).items())
    return { "summary": f"Summary:\n{bullets}" }

In [ ]:
# Build the graph — same structure as Interrupts.ipynb but with AUTOMATIC verification instead of human input.
#
# Flow:
#   START -> [wikipedia, news, arxiv] (parallel fan-out)
#   wikipedia, news -> verifier -> merge
#   arxiv -> merge  (no verification needed)
#   merge -> END
graph_builder = StateGraph(ResearchState)
graph_builder.add_node("wikipedia", wikipedia)
graph_builder.add_node("news", news)
graph_builder.add_node("arxiv", arxiv)
graph_builder.add_node("verifier", verify)
graph_builder.add_node("merge", merge)

# NOTE: The three nodes ("wikipedia", "news", "arxiv") will be executed in parallel, then two of them will be verified and only after that they will be merged together.
graph_builder.add_edge(START, "wikipedia")
graph_builder.add_edge(START, "news")
graph_builder.add_edge(START, "arxiv")

# Wikipedia and news route through the automated verifier
graph_builder.add_edge("wikipedia", "verifier")
graph_builder.add_edge("news", "verifier")
graph_builder.add_edge("verifier", "merge")

# ArXiv goes straight to merge (no verification)
graph_builder.add_edge("arxiv", "merge")

graph_builder.add_edge("merge", END)

graph = graph_builder.compile(checkpointer=checkpointer)

In [ ]:
# Visualize the graph — compare with Part 1 to see the added 'verifier' node
display_graph(graph, Path("/content/graph.png"))

In [ ]:
# Run the graph — the automated verifier will approve or reject sources without any human input
thread1_config = { "configurable": { "thread_id": "thread_1" } }
thread1_result = graph.invoke(
    input={
        "topic": "Maximum snooker break"
    },
    config=thread1_config
)

In [ ]:
# Print the final summary — sources will be labelled (verified) or (blocked) based on the rule
print(thread1_result['summary'])

In [ ]:
# Inspect the raw result dictionary — check the 'verified' and 'blocked' sets
thread1_result

In [ ]:
# Walk through the step-by-step state history to see how verification changed the state
explore_state_history(graph, thread1_config)